# QualityPhys - Camera Remote Vital Signs Estimator (CRVSE) Project

## Notebook P3-40: Respiratory rate from video - the confirming run on VitalVideos-WW's held-out split

### What this notebook does
Tests one readout under a rule fixed before any frame of the held-out split was read for breathing: whether the rate that readout A of NB_P3_39 takes from the motion of the torso, in the whole frame, lands within 2 breaths/min of a waist belt recorded with the video often enough to clear a bar set in advance, and more often than a constant at the training median. A compression ladder on the 11 held-out recordings kept as raw video asks whether a browser-grade encode moves the reading. Part 0 is that rule, committed before the run, with the facts it rests on and a freeze record that checks the code against it. The run reads the held-out split once, through `app/live_vitals/scripts/check_rr.py`; the later sections verify its provenance, apply the rule and state the outcome. No frame is rendered, and no recording appears beside its skin type, site or age.

### Readouts
- **A, pixel components** (`app/live_vitals/signal/respiration.py`, `torso_rate`) - the rows below the padded face box, in grey; each pixel's series band-passed; the first five principal components each read for their spectral peak; the rate of the most concentrated. The finding is about this readout.
- **B, vertical shift** (`check_rr.vertical_shift`) - the same rows matched against their mean by cross-correlation. Described beside A, deciding nothing.

### Set and reference
VitalVideos-WW's held-out split (VV-60): 60 participants, one 60 s recording each, seated and still, breathing spontaneously; lossless lab video at 60 fps whose framing includes face, chest and shoulders, stored as `frames_full`, the whole frame at 132x120 and 10 fps. The reference is the respiration belt worn at the waist, recorded at 500 Hz on the ECG's clock and read as `rr_belt` (NB_P3_27), scored only where an independent surrogate confirms it (`resp_ok`): 56 of the 60.

### Background
NB_P3_39 developed the readouts on the 208 `resp_ok` recordings of the training split (VV-240). Of six camera readouts only the torso's passed its development gate: A and B each read all 208 and put 197 within 2 breaths/min of the belt (94.7%, 95% 90.8-97.0%), against 71 for the training median (McNemar 128 against 2). Head motion in the face crop, and the colour and pulse modulation of the face, did not beat that constant. At the gate (2026-11-06) A became the primary readout and B a description, with no refusal rule; the app's code reproduces A's and B's 240 development readings exactly. The app keeps only the 72x72 face crop today, so the torso is out of its reach unless its capture keeps a whole-frame stream: a confirmed result here is a finding about a method, not a capability of the app.

## Part 0 - Pre-registration (2026-11-06)

Fixed after NB_P3_39's development gate and before any frame of the held-out split was read for breathing, and committed before the run. The run happens once; its code is tested on synthetic data and on the training split. **Anything that changes after the run starts makes it exploratory**, reported as such with the change and its reason. Until the run starts, a change to Part 0 is made only as a dated amendment cell after the freeze record, stating what changed and why; the text above it stays as committed. The bar and the ladder's rule were agreed on 2026-11-04, before development, and development changed neither.

**Question.** Does readout A, applied unchanged to VV-60's 56 recordings with a confirmed belt, read the breathing rate within 2 breaths/min of the belt often enough to clear a bar fixed in advance, and more often than a constant at the training median? Does a browser-grade encode move its reading?

**Scope.** A finding about a method on one dataset: seated adults at rest, breathing spontaneously, lossless lab video with the chest in frame. It is not a capability of the app, whose respiratory-rate tile stays out; an app-level claim needs a capture that keeps the whole frame, its own rule and a browser capture. No clinical reading of the number (normal ranges, early-warning scores, alarms, apnoea, health status). Site and skin type are confounded in this set (the outdoor CH* site is Fitzpatrick 4-6), so no stratum separates them. Retrospective data only; nothing trains; anything published is aggregates only. Not a medical device.

### Data and sets
- **Recordings.** The 60 VV-60 groups of `vitalvideos_phase3_rc_none.h5`, read-only, through `frames_full` exactly as stored (600 frames, 132x120, 8 bits per channel, 10 fps) and the lower edge of the stored padded face box (`box_ry1`).
- **Scored set.** The 56 with `resp_ok`. The 4 without a confirmed belt are reported against `rr_surrogate` where it exists (2 of them) and never scored.
- **The ladder.** The 11 `resp_ok` recordings among the 12 held-out participants kept as raw video (`D:\QualityPhys\demo_data\vitalvideos_subset`), each in six versions written on 2026-11-01 for NB_P3_38's check 2 by a standalone tool from commit 4173631 and recorded file by file, with byte counts and SHA-256, in the ladder's manifest. The twelfth participant (no confirmed belt) is read the same way and described.
- **The second camera.** The same 12 participants' iPhone 15 Pro recordings (ProRes 422 HQ, 1080x1920 portrait, 59.96 fps, framing face, neck and upper chest), described only.

### Reference
`rr_belt`: the belt's spectral peak over the whole minute within 6-42 breaths/min, band-edge peaks refused (NB_P3_27), reproduced from the store's 500 Hz signal to 4e-15 breaths/min (NB_P3_39, C0). `resp_ok`: the belt within 1 breath/min of ECG-derived respiration or of the contact PPG's pulse amplitude. On VV-60's 56 the belt and ECG-derived respiration agree within 2 breaths/min in 94.6% (NB_P3_39, C0).

### Procedure, frozen
- **Readout A**, as committed in 5f76c8c (`respiration.py`; SHA-256 of its text with LF line endings `f9f31fa0...efc2`). Grey rows from `ceil(box_ry1 x 132)` to the bottom of the frame, at least 8 rows and 16 frames; each pixel's series linearly detrended and band-passed to 0.08-1.0 Hz (Butterworth, order 2, zero phase); the first 5 principal components, from the frames' Gram matrix; each read by NB_P3_27's spectral readout (Hann window, zero padding to 8 times the length, log-parabolic refinement, 6-42 breaths/min, band-edge peaks refused); the reading is the rate of the component with the highest peak concentration, the share of its 6-42 breaths/min power within 0.03 Hz of its peak. One rate per recording, over the full 60 s at 10 fps.
- **No refusal rule and no confidence threshold.** Every rate A gives counts as read; a recording it does not read counts as wrong in the paired test.
- **Readout B**, as committed in `check_rr.py`, on the same rows; described.
- **The ladder's frames.** Each version is decoded as NB_P3_27 decoded the raw video (OpenCV, BGR to RGB), reduced with `INTER_AREA` from 480x528 to 120x132 and averaged over each 3 consecutive frames, rounded to 8 bits: 600 frames at 10 fps, read by A with the store's `box_ry1`. The ladder holds every second frame of the original, so each average covers frames 0, 2 and 4 of the six that one frame of `frames_full` averages.
- **The second camera's frames.** Decoded the same way (OpenCV, which applies the file's rotation), reduced 8-fold with `INTER_AREA` to 135x240 and averaged over each 6 consecutive frames: 599 frames at 9.99 fps. The face box comes from the app's own detector - `stable_box` on the 12 samples `detection_positions` picks - and its lower edge, as a fraction of the height and clamped to the frame, takes the place of `box_ry1`.

### The rule
- **Measure.** Per recording, A's rate against `rr_belt`. A recording is *read* when A gives a rate, and *right* when that rate lies within 2 breaths/min of the belt. The share is right over read; coverage is read over 56.
- **The baseline.** 19.94 breaths/min, the median belt rate of VV-240's 208 `resp_ok` recordings (NB_P3_39, C0), given to every recording and right when within 2 breaths/min. On VV-60's 56 it is right in 22 (39.3%).
- **The paired test.** Exact McNemar: a two-sided binomial test at one half on the discordant recordings, those where exactly one of A and the baseline is right; a recording A does not read counts as A wrong.
- **Interval.** Wilson, two-sided 95% (z = 1.959964).
- **The three outcomes:**
  - *confirmed* when the interval's lower bound is at least 70%, coverage is at least 80% (45 of 56), and A is ahead on the discordant recordings with p < 0.05;
  - *not confirmed* when the share is below 70%, or the paired test is not passed (p of 0.05 or more, or A not ahead);
  - *inconclusive* otherwise - reported as such and never reframed; it changes nothing in the app.

  With all 56 read, 46 right confirm (82.1%; 45 give a lower bound of 68.2%), and 39 or fewer are not confirmed (69.6%); 40-45 are inconclusive if the paired test passes. With 45 read, 38 confirm.

### The ladder's rule
- **The rungs.** Every rung starts from the same frames: every second frame of the original (1800 at 30 fps), scaled to half size (480x528) with `INTER_AREA`, plane by plane in the source's YUV 4:4:4. **R0** - lossless 4:4:4 (libx264, quantiser 0), the baseline; **R1** - lossless 4:2:0; **R2** - H.264 4:2:0 at 0.651 bits per pixel per frame, the app's recorder setting (640x480 at 6 Mbit/s and 30 fps), here 4.95 Mbit/s: **the deciding rung**; **R3** and **R4** - H.264 at a half and a quarter of that; **R5** - VP8 at R2's rate. The lossy rungs imitate a real-time encoder (libx264 `veryfast` with `zerolatency`; VP8 `realtime`), which is not a browser's encoder.
- **Self-test, first.** On each of the 12 recordings, R0 reduced as above has `frames_full`'s shape, and over A's torso region the two differ by a mean absolute 1.0 grey level or less once each pixel's mean over the minute is removed. On VV-240's 60 fps face crops a 3-of-6-frame mean lies a mean 0.10-0.18 levels from the 6-frame mean (99th percentile 1.0; 10 recordings). If any recording fails, the ladder is *not assessed* and the sentence says so; the verdict on the stored frames stands.
- **Measure.** Per recording, A's reading at R2 against its reading at R0, each recording its own baseline; no belt is involved.
- **Outcomes**, on the 11 `resp_ok` recordings:
  - *no effect* when every recording A reads at R0 is also read at R2, within 1.0 breath/min of its R0 reading;
  - *effect* when at least 2 of the 11 lose their reading at R2 or move by more than 2.0 breaths/min;
  - *inconclusive* otherwise.
- **What it decides.** Only how a confirmed result reaches the app: with an *effect* it is stated as lab-only and the app's tile stays out; with *no effect* the capture question - whether the app keeps a whole-frame stream - opens in the app's own work; *inconclusive* or *not assessed* leave that question open. The ladder never changes the verdict.
- R1, R3, R4 and R5 are judged the same way against R0 and decide nothing; so are the twelfth recording and the iPhone recordings.

### Hypotheses, stated in advance
Judged on point estimates; the intervals are reported beside them.
- **H1.** A confirms: at least 46 of 56 within 2 breaths/min. High confidence; at VV-240's share the expected count is 53.
- **H2.** A reads all 56 (VV-240: 208 of 208).
- **H3.** The ladder shows no effect at R2: torso motion is slow and spatially coarse, and the reduction to 120x132 averages most coding noise away. Medium confidence.
- **H4.** A's misses sit in its least concentrated recordings: at least half of them below VV-240's lowest quintile cut of peak concentration (0.561).
- **H5.** The iPhone recordings read within 2 breaths/min of the belt in at least 8 of the 11 `resp_ok` participants. Low confidence: the framing differs, and the padded box may leave little torso below it.

### Reported, deciding nothing
- **For A and B:** within 1 breath/min, the mean absolute error, the bias and the 95% limits of agreement; B's share, coverage and paired test beside A's, never substituted for them; A and B against each other (both right, either alone, neither).
- **Strata**, each stratum's right over read: belt rate band (8 or fewer, 9-11, 12-20, 21-24, 25 or more breaths/min; VV-60 holds 3, 1, 35, 13 and 4), site group, Fitzpatrick pair (1-2, 3-4, 5-6) and age band (40 or under, 41-60, over 60). `resp_ok` leans young: on VV-60 it holds 17 of 18 participants aged 40 or under, 30 of 30 aged 41-60 and 9 of 12 over 60. No stratum decides, and none is named as a finding.
- **Peak concentration:** A's right over read in the five bands cut at VV-240's quintiles of A's concentration (0.561, 0.707, 0.802, 0.864; full precision in the freeze record).
- **Without a confirmed belt:** the 4 recordings, against `rr_surrogate` where it exists (2).
- **The ladder:** every rung's readings against R0 with the rule's counts, the twelfth recording, the self-test's figures, and R0's readings against those on the stored frames.
- **The second camera:** the iPhone recordings' readings against the belt and against the stored frames' reading, and the rows each leaves below its box.

### The sentence
Its wording is fixed now; the run fills the slots, and the verb follows from the outcome. Model card (the README takes the first clause and a link):

> Respiratory rate from the motion of the torso, seated adults at rest. On the [n] of 60 held-out VitalVideos-WW recordings whose waist belt an independent surrogate confirms (lossless lab video with the chest in frame, not a browser capture), readout A [reads / was neither shown to read nor shown not to read / does not read] the breathing rate within 2 breaths/min of the belt often enough to clear a bar fixed in advance[, on lossless lab video only]: [k] of the [r] recordings it reads ([share]%, 95% interval [lo]-[hi]%; within 1 breath/min [w1]%), coverage [c]%, against [b]% for a constant at the training median (exact McNemar p = [p]). A browser-grade encode (H.264 at 0.65 bits per pixel per frame, 11 recordings) [did not move the reading / moved the reading / had an effect that stayed unresolved / was not assessed]. Developed on 208 training recordings (94.7% within 2 breaths/min), where no face-only readout beat the constant. Research use only: not a medical device, and no clinical reading of the rate.

The slots: [n] the scored set; the verb *reads*, *was neither shown to read nor shown not to read* or *does not read* for confirmed, inconclusive or not confirmed; "on lossless lab video only" only with a ladder *effect*; [k], [r], [share] and [lo]-[hi] A's right, read, share and Wilson interval; [w1] within 1 breath/min; [c] coverage; [b] the baseline's share; [p] the paired test; the ladder's clause for *no effect*, *effect*, *inconclusive* or *not assessed*. Never dropped: the condition, n and coverage, the share and its interval, the baseline, the ladder's clause, the development line, the lossless-video caveat and the research-use line.

### Preflight
The run's outputs are scored only if every check passes; any failure stops the notebook.
1. The run's summary names one commit with no uncommitted app code; between Part 0's commit and that commit nothing under `app/` changes except `check_rr.py` and its test file, and `respiration.py` still hashes to `f9f31fa0...efc2`.
2. Before reading any held-out frame, the run reproduces NB_P3_39's development readings at that commit: digest `f75095c4a475f6d8` and A's counts; otherwise it stops.
3. No `--limit`: all 60 VV-60 recordings, all 72 ladder files, all 12 iPhone recordings.
4. Every ladder file matches the manifest's byte count and SHA-256 before it is read.
5. The settings the run records equal the freeze record's.
6. The ladder's self-test is reported before any rung's reading.

### Interpretation fixed in advance
A confirmed result means that on this dataset's lossless recordings, with the chest in frame and the participants seated and still, readout A's 60 s breathing rate lands within 2 breaths/min of a waist belt often enough to clear the bar, and more often than a constant. It does not mean the app measures breathing rate: the app keeps only the face crop, its captures are compressed, and an app-level claim needs its own rule. A result that is not confirmed closes respiratory rate for this phase as a documented negative; an inconclusive one is written up as such and changes nothing in the app. After a confirmed result, and not before: the question of a whole-frame capture, and an independent webcam-grade test (COHFACE).

### On record before the freeze
**Development (NB_P3_39, VV-240's 208 `resp_ok` recordings).** A: read 208, within 2 breaths/min 197 (94.7%, 95% 90.8-97.0%), within 1 185 (88.9%); mean absolute error 0.61, bias -0.32, 95% limits of agreement -4.39 to +3.74; McNemar 128 against 2. B: 197 (94.7%), within 1 188 (90.4%); 0.73, -0.45, -6.19 to +5.28. At 25 breaths/min or more A was right in 29 of 29 and B in 23 of 29. A's least concentrated fifth held 10 of its 11 misses (32 of 42 right, against 165 of 166 in the rest). The two were right together in 188. Head motion in the face crop (C2): 69 of 192 (35.9%). The face's baseline intensity, pulse amplitude and pulse-rate swing (C3): 16.9%, 24.9% and 34.8%, none better than the constant. On Rice's paced deep breathing (C4) the pulse-rate swing on POS's beats followed the finger's in 13 of 13, intensity and amplitude in 4 and 5.

**The gate (2026-11-06).** A primary; no second hypothesis; no refusal rule; no other dataset before this verdict; learned models stay closed (NB_P3_39, section 11).

**The app's code.** `signal/respiration.py`, `scripts/check_rr.py` and their 25 tests, committed in 5f76c8c. The development stage reproduces A's and B's 240 readings exactly (digest `f75095c4a475f6d8`) in `Data/rr_eval/summary_dev.json`, committed in 2969111; it ran on the working tree before the code's commit (its provenance names 500ecc8 with uncommitted changes), and the module hash it records equals the committed file's.

**VV-60, camera-free.** 60 groups, each with `frames_full` of 600 x 132 x 120 x 3 at 10 fps and a torso region of 10-74 rows; 56 `resp_ok`; the baseline right in 22 (39.3%); belt rate bands 3 / 1 / 35 / 13 / 4; belt and ECG-derived respiration within 2 breaths/min in 94.6%. The 11 ladder recordings: belt rates 8.1-23.4 breaths/min (none at 25 or more), 4 CH*, 4 MOL/RING and 3 CRC/SCH/BET, torso regions of 28-62 rows.

**Seen before the freeze.**
- No respiratory readout has touched a frame of VV-60, of the ladder or of the iPhone recordings.
- VV-60's belt values, surrogates and attributes (above), read to score the baseline and size the sets.
- The raw and iPhone recordings were decoded earlier for heart rate and framing only: NB_P3_38's check 2 read the face crop on every ladder rung on 2026-11-01 (v2 read 10 of 12 at R0, and at R2 kept 9 of those 10 within 0.72 bpm), and the app's framing gate refuses every iPhone recording (11.5-23% of the box's side outside the frame). Nothing about breathing was computed on them.
- The self-test's tolerance comes from VV-240's 60 fps face crops (every 24th training recording by GUID, 10 in all), not from a held-out frame.
- The nine VV-240 recordings probed on 2026-11-03 (NB_P3_39).

### Freeze record
Checks that the code at this commit carries readout A as written above - its constants and the hash of its text - and that `check_rr.py` carries the development measure; reads the development summary and its per-recording table back and verifies that they reproduce NB_P3_39 with this module; and checks the data the run will read without reading a frame: VV-60's 60 groups with `frames_full` in the stated shape and rate and 56 `resp_ok`; the ladder's manifest (72 files, the recipe above, written from commit 4173631 with a clean tree) and its files present at the stated sizes; the 12 iPhone recordings. It records the numbers the run implements, the bar's arithmetic, VV-240's concentration cuts at full precision, and a fingerprint of the ladder and iPhone files - the SHA-256 of their sorted names and sizes - so that the run can show it read the same files. The cell stops if any check fails.

In [1]:
import hashlib
import json
import sys
from pathlib import Path

import h5py
import numpy as np
import pandas as pd


def find_repo_root(start):
    '''The first folder at or above `start` holding both app/ and Data/.'''
    for folder in [start, *start.parents]:
        if (folder / 'app').is_dir() and (folder / 'Data').is_dir():
            return folder
    raise FileNotFoundError(f'no repository root at or above {start}')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from app.live_vitals.scripts import check_rr
from app.live_vitals.signal import respiration

# Part 0's values as written above. Each must equal the code at this commit.
FROZEN = {
    'frame rate (fps)': 10.0,
    'read band (Hz)': (0.10, 0.70),
    'zero padding (times the length)': 8,
    'pixel band-pass (Hz)': (0.08, 1.0),
    'band-pass order': 2,
    'least torso rows': 8,
    'least frames': 16,
    'principal components': 5,
    'concentration half-width (Hz)': 0.03,
    'within (breaths/min)': 2.0,
}
IN_CODE = {
    'frame rate (fps)': respiration.FULL_FS,
    'read band (Hz)': (respiration.RESP_LOW_HZ, respiration.RESP_HIGH_HZ),
    'zero padding (times the length)': respiration.FFT_PAD,
    'pixel band-pass (Hz)': respiration.BREATH_BAND_HZ,
    'band-pass order': respiration.BREATH_FILTER_ORDER,
    'least torso rows': respiration.MIN_TORSO_ROWS,
    'least frames': respiration.MIN_FRAMES,
    'principal components': respiration.N_COMPONENTS,
    'concentration half-width (Hz)': respiration.SHARE_HALF_WIDTH_HZ,
    'within (breaths/min)': check_rr.WITHIN_BPM,
}
MODULE_SHA256_LF = 'f9f31fa03882bba2b338646c305027ccd65ac5529dd2eff3d6fb396a2734efc2'
CODE_COMMIT = '5f76c8c77f07779bb467d4b4206f8da8203670eb'
NB39_DIGEST = 'f75095c4a475f6d8'
BROWSER_BITS = 6e6 / (640 * 480 * 30)
# Fixed here; the run implements them.
RULE = {
    'bar: Wilson 95% lower bound at least': 0.70,
    'bar: coverage at least': 0.80,
    'bar: McNemar p below': 0.05,
    'Wilson z': 1.959964,
    'ladder: deciding rung': 'R2',
    'ladder: no effect, every move within (breaths/min)': 1.0,
    'ladder: effect, a move beyond (breaths/min)': 2.0,
    'ladder: effect, least recordings lost or moved': 2,
    'ladder: downscale from half size, frames averaged': (4, 3),
    'self-test: mean absolute difference, pixel means removed (grey levels)': 1.0,
    'iPhone: downscale, frames averaged': (8, 6),
}
STORE = check_rr.DEFAULT_STORE
DEV_DIR = REPO_ROOT / 'Data' / 'rr_eval'
LADDER_DIR = Path(r'D:\QualityPhys\demo_data\vitalvideos_reencoded')
LADDER_COMMIT = '41736314e7dafd7799b31ec97c9dd0452825f2cc'
RAW_SUBSET = Path(r'D:\QualityPhys\demo_data\vitalvideos_subset')
RUNGS = {'R0': ('libx264', 'yuv444p', None), 'R1': ('libx264', 'yuv420p', None),
         'R2': ('libx264', 'yuv420p', BROWSER_BITS), 'R3': ('libx264', 'yuv420p', BROWSER_BITS / 2),
         'R4': ('libx264', 'yuv420p', BROWSER_BITS / 4), 'R5': ('libvpx', 'yuv420p', BROWSER_BITS)}

problems = [f'{name}: Part 0 {value!r}, code {IN_CODE[name]!r}'
            for name, value in FROZEN.items() if IN_CODE[name] != value]
module_hash = check_rr.text_sha256(respiration.__file__)
if module_hash != MODULE_SHA256_LF:
    problems.append(f'respiration.py hashes to {module_hash}')

# The development summary and its local per-recording table: the module reproduced NB_P3_39.
dev = json.loads((DEV_DIR / 'summary_dev.json').read_text(encoding='utf-8'))
repro = dev['reproduction']
if not (repro['checked'] and repro['reproduces'] and repro['digest'] == NB39_DIGEST
        and dev['limit'] is None and dev['recordings'] == 240
        and dev['module']['sha256_lf'] == MODULE_SHA256_LF):
    problems.append(f"development summary: {repro}, module {dev['module']['sha256_lf']}")
dev_rows = pd.read_csv(DEV_DIR / 'recordings_dev.csv', float_precision='round_trip')
if len(dev_rows) != 240 or check_rr.readings_digest(dev_rows)[:16] != NB39_DIGEST:
    problems.append(f'recordings_dev.csv: {len(dev_rows)} rows, digest '
                    f'{check_rr.readings_digest(dev_rows)[:16]}')
BASELINE = dev['baseline_bpm']
CONCENTRATION_CUTS = np.quantile(dev_rows.loc[dev_rows.resp_ok, 'a_share'], [0.2, 0.4, 0.6, 0.8]).tolist()

# VV-60 without reading a frame: attributes and dataset shapes only.
with h5py.File(STORE, 'r') as store:
    held_out = sorted(g for g in store if str(store[g].attrs['split']) == 'val')
    geometry = {(store[g]['frames_full'].shape, float(store[g].attrs['full_fps'])) for g in held_out}
    resp_ok = sorted(g for g in held_out if bool(store[g].attrs['resp_ok']))
if len(held_out) != 60 or geometry != {((600, 132, 120, 3), 10.0)} or len(resp_ok) != 56:
    problems.append(f'VV-60: {len(held_out)} groups, geometry {geometry}, {len(resp_ok)} resp_ok')

# The ladder: its manifest, its recipe and its files' sizes; the raw subset's iPhone recordings.
raw_guids = sorted(p.name for p in RAW_SUBSET.iterdir() if p.is_dir())
ladder_guids = [g for g in raw_guids if g in resp_ok]
if len(raw_guids) != 12 or not set(raw_guids) <= set(held_out) or len(ladder_guids) != 11:
    problems.append(f'raw subset: {len(raw_guids)} participants, {len(ladder_guids)} resp_ok')
manifest = json.loads((LADDER_DIR / 'manifest.json').read_text(encoding='utf-8'))
recipe = manifest['recipe']
if (manifest['repository']['commit'] != LADDER_COMMIT or manifest['repository']['dirty']
        or sorted(manifest['recordings']) != raw_guids or recipe['frame_step'] != 2
        or (recipe['ladder']['height'], recipe['ladder']['width'], recipe['ladder']['fps']) != (528, 480, 30)
        or recipe['source'] != dict(codec='h264', pix_fmt='yuv444p', height=1056, width=960, fps=60.0, frames=3600)
        or {r['rung']: (r['codec'], r['pix_fmt'], r['bits']) for r in recipe['rungs']} != RUNGS):
    problems.append(f"ladder manifest: commit {manifest['repository']}, recipe {recipe}")
ladder_files = []
for name, entry in sorted(manifest['files'].items()):
    path = LADDER_DIR / name
    size = path.stat().st_size if path.is_file() else None
    if size != entry['bytes']:
        problems.append(f'ladder file {name}: {size} bytes, manifest {entry["bytes"]}')
    ladder_files.append((name, size))
if len(ladder_files) != 72:
    problems.append(f'the ladder manifest lists {len(ladder_files)} files')
iphone = sorted(RAW_SUBSET.glob('*/*.MOV'))
if len(iphone) != 12 or sorted(p.parent.name for p in iphone) != raw_guids:
    problems.append(f'{len(iphone)} iPhone recordings under {RAW_SUBSET}')
iphone_files = [(f'{p.parent.name}/{p.name}', p.stat().st_size) for p in iphone]
FILES_FINGERPRINT = hashlib.sha256(''.join(f'{n} {s}\n' for n, s in ladder_files + iphone_files)
                                   .encode('utf-8')).hexdigest()

# The bar's arithmetic at the coverages that can confirm.
ARITHMETIC = {n: (min(k for k in range(n + 1) if check_rr.wilson(k, n)[0] >= RULE['bar: Wilson 95% lower bound at least']),
                  max(k for k in range(n + 1) if k / n < RULE['bar: Wilson 95% lower bound at least']))
              for n in (56, 50, 45)}

if problems:
    raise SystemExit('Freeze record FAILED:\n  ' + '\n  '.join(problems))
print('Freeze record - Part 0 of NB_P3_40')
print(f'  readout A         : {len(FROZEN)} settings equal the code; respiration.py SHA-256 (LF) '
      f'{module_hash[:8]}...{module_hash[-4:]}, committed in {CODE_COMMIT[:7]}')
print(f"  development       : summary_dev.json reproduces NB_P3_39 (digest {repro['digest']}, A "
      f"{repro['a_counts']['within_2']}/{repro['a_counts']['read']}), its table too; baseline {BASELINE:.6f} breaths/min")
print(f'  VV-60             : {len(held_out)} groups, frames_full 600 x 132 x 120 x 3 at 10 fps, {len(resp_ok)} resp_ok')
print(f'  ladder            : {len(ladder_files)} files, {sum(s for _, s in ladder_files) / 1e9:.2f} GB, '
      f'written from {LADDER_COMMIT[:7]} (clean); {len(ladder_guids)} resp_ok participants of {len(raw_guids)}')
print(f'  iPhone            : {len(iphone_files)} recordings, {sum(s for _, s in iphone_files) / 1e9:.1f} GB')
print(f'  files fingerprint : {FILES_FINGERPRINT}')
print('fixed here, implemented by the run:')
for name, value in RULE.items():
    print(f'  {name:70} {value}')
print('the bar at n read: least right that confirm | most right not confirmed')
for n, (confirm, below) in ARITHMETIC.items():
    print(f'  {n}: {confirm} ({confirm / n:.1%}) | {below} ({below / n:.1%})')
print(f'VV-240 concentration cuts (quintiles of A\'s share): {", ".join(f"{c:.6f}" for c in CONCENTRATION_CUTS)}')

Freeze record - Part 0 of NB_P3_40
  readout A         : 10 settings equal the code; respiration.py SHA-256 (LF) f9f31fa0...efc2, committed in 5f76c8c
  development       : summary_dev.json reproduces NB_P3_39 (digest f75095c4a475f6d8, A 197/208), its table too; baseline 19.939173 breaths/min
  VV-60             : 60 groups, frames_full 600 x 132 x 120 x 3 at 10 fps, 56 resp_ok
  ladder            : 72 files, 9.72 GB, written from 4173631 (clean); 11 resp_ok participants of 12
  iPhone            : 12 recordings, 38.4 GB
  files fingerprint : 37295672f706a15b1460a0988d5b84d1d835e6f9ff11631b475095eace539166
fixed here, implemented by the run:
  bar: Wilson 95% lower bound at least                                   0.7
  bar: coverage at least                                                 0.8
  bar: McNemar p below                                                   0.05
  Wilson z                                                               1.959964
  ladder: deciding rung             

## 1. The run's provenance
The held-out stage ran once, through `app/live_vitals/scripts/check_rr.py --stage confirm`. Its summary is `Data/rr_eval/summary_confirm.json`; its per-recording tables (`recordings_confirm.csv`, `ladder_confirm.csv`, `iphone_confirm.csv`) stay local. This cell applies Part 0's preflight: the run's commit and what changed under `app/` since Part 0's, the development readings reproduced before any held-out frame was read, the run complete, the files it read, its settings against the freeze record, and the self-test reported before the rungs. Any failure stops the notebook before anything is scored.

In [2]:
import math
import subprocess
import textwrap


def git(*args):
    '''A git command's output in the repository, as text.'''
    return subprocess.run(['git', *args], cwd=REPO_ROOT, capture_output=True, text=True, check=True).stdout


def plain(value):
    '''A value as JSON would hold it, so that tuples and lists compare equal.'''
    return json.loads(json.dumps(value))


OUT_DIR = REPO_ROOT / 'Data' / 'rr_eval'
RUN = json.loads((OUT_DIR / 'summary_confirm.json').read_text(encoding='utf-8'))
NOTEBOOK = 'Notebooks/Phase 3 Notebooks/NB_P3_40_RR_Confirmation.ipynb'
PART0_COMMIT = git('log', '--diff-filter=A', '--format=%H', '--', NOTEBOOK).split()[-1]
ALLOWED = {'app/live_vitals/scripts/check_rr.py', 'app/live_vitals/tests/test_respiration.py'}
# What Part 0 and its freeze record say the run must show.
EXPECTED = dict(recordings=60, scored=56, ladder_recordings=12, ladder_resp_ok=11, ladder_files=72,
                iphone_files=12, fingerprint=FILES_FINGERPRINT, ladder_commit=LADDER_COMMIT)

run_git = RUN['provenance']['git']
run_commit = run_git['commit']
ancestor = subprocess.run(['git', 'merge-base', '--is-ancestor', PART0_COMMIT, run_commit],
                          cwd=REPO_ROOT).returncode == 0
changed = set(git('diff', '--name-only', PART0_COMMIT, run_commit, '--', 'app').split())
module_at_run = hashlib.sha256(subprocess.run(
    ['git', 'show', f'{run_commit}:app/live_vitals/signal/respiration.py'], cwd=REPO_ROOT,
    capture_output=True, check=True).stdout.replace(b'\r\n', b'\n')).hexdigest()
dev_run, files, ladder_run = RUN['development'], RUN['files'], RUN['ladder']
CHECKS = [
    ('1. one commit, no uncommitted app code',
     not any(path.startswith('app/') for path in run_git['changed']), f"{run_commit[:7]}, changed {run_git['changed']}"),
    ("1. Part 0's commit precedes the run's", ancestor, f'{PART0_COMMIT[:7]} -> {run_commit[:7]}'),
    ('1. app/ changed only in check_rr.py and its tests', changed <= ALLOWED, sorted(changed)),
    ('1. respiration.py unchanged', module_at_run == MODULE_SHA256_LF == RUN['module']['sha256_lf'],
     f'{module_at_run[:8]}...{module_at_run[-4:]}'),
    ('2. development readings reproduced first',
     dev_run['reproduces'] and dev_run['digest'] == NB39_DIGEST and dev_run['a_counts'] == check_rr.NB39_A_COUNTS,
     f"digest {dev_run['digest']}, A {dev_run['a_counts']['within_2']}/{dev_run['a_counts']['read']}"),
    ('3. no limit, every recording',
     RUN['limit'] is None and RUN['held_out']['recordings'] == EXPECTED['recordings']
     and RUN['held_out']['scored'] == EXPECTED['scored'] and ladder_run['recordings'] == EXPECTED['ladder_recordings']
     and ladder_run['resp_ok'] == EXPECTED['ladder_resp_ok'] and RUN['iphone']['recordings'] == EXPECTED['iphone_files'],
     f"VV-60 {RUN['held_out']['recordings']} ({RUN['held_out']['scored']} scored), ladder {ladder_run['recordings']} "
     f"({ladder_run['resp_ok']} resp_ok), iPhone {RUN['iphone']['recordings']}"),
    ('4. the files the freeze record fingerprinted',
     files['fingerprint'] == EXPECTED['fingerprint'] and files['ladder_commit'] == EXPECTED['ladder_commit']
     and files['ladder_files'] == EXPECTED['ladder_files'] and files['iphone_files'] == EXPECTED['iphone_files'],
     f"fingerprint {files['fingerprint'][:16]}..., {files['ladder_files']} ladder files"),
    ("5. settings equal the freeze record's", plain(RUN['settings']) == plain(RULE), f'{len(RULE)} settings'),
    ('6. the self-test reported before the rungs',
     'passed' in ladder_run['self_test'] and (ladder_run['self_test']['passed'] or not ladder_run['rungs']),
     f"passed {ladder_run['self_test'].get('passed')}"),
]
for name, ok, detail in CHECKS:
    print(f"  {'ok    ' if ok else 'FAILED'} {name}: {detail}")
failed = [name for name, ok, _ in CHECKS if not ok]
if failed:
    raise SystemExit('Preflight FAILED: ' + '; '.join(failed))
print(f"Preflight passed: the run of {RUN['provenance']['written_utc']} at {run_commit[:7]} is scored.")

  ok     1. one commit, no uncommitted app code: 495e2d6, changed []
  ok     1. Part 0's commit precedes the run's: 048483b -> 495e2d6
  ok     1. app/ changed only in check_rr.py and its tests: ['app/live_vitals/scripts/check_rr.py', 'app/live_vitals/tests/test_respiration.py']
  ok     1. respiration.py unchanged: f9f31fa0...efc2
  ok     2. development readings reproduced first: digest f75095c4a475f6d8, A 197/208
  ok     3. no limit, every recording: VV-60 60 (56 scored), ladder 12 (11 resp_ok), iPhone 12
  ok     4. the files the freeze record fingerprinted: fingerprint 37295672f706a15b..., 72 ladder files
  ok     5. settings equal the freeze record's: 11 settings
  ok     6. the self-test reported before the rungs: passed True
Preflight passed: the run of 2026-11-06T17:15:12+00:00 at 495e2d6 is scored.


## 2. The rule and the outcome
A's and B's figures, the outcome and the ladder's rule are recomputed from the local per-recording tables, and each must equal the summary's; Part 0's bar then gives the outcome, the ladder's rule its clause, and the sentence is filled as Part 0 wrote it.

In [3]:
def same(a, b, rel=1e-12):
    '''Equal structures, floats to a relative 1e-12 and NaN equal to NaN.'''
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(same(a[k], b[k], rel) for k in a)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(same(x, y, rel) for x, y in zip(a, b))
    if isinstance(a, float) or isinstance(b, float):
        if a is None or b is None:
            return a is b
        return (math.isnan(a) and math.isnan(b)) or math.isclose(a, b, rel_tol=rel, abs_tol=1e-15)
    return a == b


recordings = pd.read_csv(OUT_DIR / 'recordings_confirm.csv', float_precision='round_trip')
ladder_rows = pd.read_csv(OUT_DIR / 'ladder_confirm.csv', float_precision='round_trip')
scored = recordings[recordings.resp_ok]
held_out = RUN['held_out']
baseline = held_out['baseline_bpm']
a = check_rr.gate(scored, 'a_rate', baseline)
b = check_rr.gate(scored, 'b_rate', baseline)
outcome = check_rr.held_out_outcome(a)
base_hits = int(((scored.belt - baseline).abs() <= check_rr.WITHIN_BPM).sum())

resp_ok_raw = sorted(ladder_rows.loc[ladder_rows.resp_ok, 'guid'].unique())
rate_at = {rung: dict(zip(rows.guid, rows.rate)) for rung, rows in ladder_rows.groupby('rung')}
r0_rows = ladder_rows[ladder_rows.rung == 'R0']
self_test_passed = bool(((r0_rows.self_test <= check_rr.SELF_TEST_LEVELS) & r0_rows.self_test.notna()).all())
rungs = {rung: check_rr.rung_outcome({g: rate_at['R0'][g] for g in resp_ok_raw},
                                     {g: rate_at[rung][g] for g in resp_ok_raw})
         for rung in check_rr.RUNGS[1:] if self_test_passed}
ladder_outcome = rungs[check_rr.DECIDING_RUNG]['outcome'] if self_test_passed else 'not assessed'

recomputed = [
    ('baseline: the development median', math.isclose(baseline, BASELINE, rel_tol=1e-12)),
    ('baseline right', base_hits == held_out['baseline_within_2']),
    ("A's figures", same(a, held_out['readouts']['A pixel components'])),
    ("B's figures", same(b, held_out['readouts']['B vertical shift, described'])),
    ('the outcome', outcome == held_out['outcome']),
    ('the self-test', self_test_passed == RUN['ladder']['self_test']['passed']),
    ("the rungs' counts and outcomes",
     all(same({k: v for k, v in rungs[r].items()}, {k: v for k, v in RUN['ladder']['rungs'][r].items()
                                                     if k != 'described_without_belt'}) for r in rungs)),
    ("the ladder's outcome", ladder_outcome == RUN['ladder']['outcome']),
]
problems = [name for name, ok in recomputed if not ok]
if problems:
    raise SystemExit('The local tables do not reproduce the summary: ' + '; '.join(problems))

lo, hi = a['wilson_95']
print(f"A on VV-60's {len(scored)} resp_ok recordings: {a['within_2']} of {a['read']} read within 2 breaths/min "
      f"({a['share']:.1%}, 95% {lo:.1%}-{hi:.1%}), coverage {a['coverage']:.1%}; McNemar against the baseline: "
      f"A only {a['readout_only']}, baseline only {a['baseline_only']}, p = {a['mcnemar_p']:.2g}")
print(f"  bar: lower bound >= {check_rr.BAR_LOWER:.0%}, coverage >= {check_rr.BAR_COVERAGE:.0%}, "
      f"A ahead with p < {check_rr.BAR_P} -> {outcome.upper()}")
print(f"The ladder at {check_rr.DECIDING_RUNG}, {len(resp_ok_raw)} resp_ok recordings: self-test "
      f"{'passed' if self_test_passed else 'FAILED'} -> {ladder_outcome.upper()}")

VERB = {'confirmed': 'reads', 'inconclusive': 'was neither shown to read nor shown not to read',
        'not confirmed': 'does not read'}
LADDER_CLAUSE = {'no effect': 'did not move the reading', 'effect': 'moved the reading',
                 'inconclusive': 'had an effect that stayed unresolved', 'not assessed': 'was not assessed'}
SENTENCE = (
    'Respiratory rate from the motion of the torso, seated adults at rest. '
    f'On the {len(scored)} of 60 held-out VitalVideos-WW recordings whose waist belt an independent surrogate '
    'confirms (lossless lab video with the chest in frame, not a browser capture), readout A '
    f'{VERB[outcome]} the breathing rate within 2 breaths/min of the belt often enough to clear a bar fixed '
    f"in advance{', on lossless lab video only' if ladder_outcome == 'effect' else ''}: "
    f"{a['within_2']} of the {a['read']} recordings it reads ({100 * a['share']:.1f}%, 95% interval "
    f"{100 * lo:.1f}-{100 * hi:.1f}%; within 1 breath/min {100 * a['within_1'] / a['read']:.1f}%), "
    f"coverage {100 * a['coverage']:.1f}%, against {100 * base_hits / len(scored):.1f}% for a constant at the "
    f"training median (exact McNemar p = {a['mcnemar_p']:.2g}). A browser-grade encode (H.264 at 0.65 bits per "
    f'pixel per frame, 11 recordings) {LADDER_CLAUSE[ladder_outcome]}. Developed on 208 training recordings '
    '(94.7% within 2 breaths/min), where no face-only readout beat the constant. Research use only: not a '
    'medical device, and no clinical reading of the rate.')
print('\nThe sentence, as Part 0 fixed it:\n' + textwrap.fill(SENTENCE, 110))

A on VV-60's 56 resp_ok recordings: 54 of 56 read within 2 breaths/min (96.4%, 95% 87.9%-99.0%), coverage 100.0%; McNemar against the baseline: A only 32, baseline only 0, p = 4.7e-10
  bar: lower bound >= 70%, coverage >= 80%, A ahead with p < 0.05 -> CONFIRMED
The ladder at R2, 11 resp_ok recordings: self-test passed -> NO EFFECT

The sentence, as Part 0 fixed it:
Respiratory rate from the motion of the torso, seated adults at rest. On the 56 of 60 held-out VitalVideos-WW
recordings whose waist belt an independent surrogate confirms (lossless lab video with the chest in frame, not
a browser capture), readout A reads the breathing rate within 2 breaths/min of the belt often enough to clear
a bar fixed in advance: 54 of the 56 recordings it reads (96.4%, 95% interval 87.9-99.0%; within 1 breath/min
91.1%), coverage 100.0%, against 39.3% for a constant at the training median (exact McNemar p = 4.7e-10). A
browser-grade encode (H.264 at 0.65 bits per pixel per frame, 11 recordings) did n

## 3. Reported, deciding nothing
As Part 0 listed them: B beside A, and the two against each other; the strata; A in the development's concentration bands; the recordings without a confirmed belt; the ladder's rungs, its twelfth recording, the self-test and R0 against the stored frames; the iPhone recordings; and the hypotheses, judged on point estimates.

In [4]:
check_rr.print_gate('A pixel components', a)
check_rr.print_gate('B vertical shift, described', b)
pair = held_out['a_against_b']
print(f"A against B, both read in {pair['both_read']}: within 2 of each other {pair['within_2_of_each_other']} | "
      f"both right {pair['both_right']}, A only {pair['a_only']}, B only {pair['b_only']}, neither {pair['neither']}")

print('\nWithin 2 breaths/min among those read (right/read, of n), described')
for label in ('site group', 'Fitzpatrick', 'age', 'belt rate'):
    cells = [f"{key} n {v['n']}: A {v['right']}/{v['read']}, B {held_out['strata']['b'][label][key]['right']}/"
             f"{held_out['strata']['b'][label][key]['read']}" for key, v in held_out['strata']['a'][label].items()]
    print(f'  {label:11s} ' + ' | '.join(cells))

cuts = held_out['concentration']['cuts']
edges = ['below'] + [f'{c:.3f}+' for c in cuts]
print("\nA by the development's concentration bands (right/read): " + ' | '.join(
    f"{edge} {band['right']}/{band['read']}" for edge, band in zip(edges, held_out['concentration']['bands'])))
unconfirmed = held_out['without_confirmed_belt']
print(f"Without a confirmed belt: {unconfirmed['recordings']}, {unconfirmed['with_surrogate']} with rr_surrogate; "
      f"A read {unconfirmed['a_read']} of those, within 2 of it {unconfirmed['a_within_2_of_surrogate']}")

ladder = RUN['ladder']
print(f"\nThe ladder: self-test {ladder['self_test']['within']} of {ladder['recordings']} within "
      f"{check_rr.SELF_TEST_LEVELS} grey level (median {ladder['self_test']['median']:.3f}, max "
      f"{ladder['self_test']['max']:.3f})")
if ladder['r0_against_stored']:
    print(f"  R0 against the stored frames: median |difference| {ladder['r0_against_stored']['median']:.2f}, "
          f"max {ladder['r0_against_stored']['max']:.2f} breaths/min, {ladder['r0_against_stored']['read_at_both']} read at both")
for rung, result in ladder['rungs'].items():
    twelfth = result['described_without_belt']
    largest = result['max_move']
    print(f"  {rung}: lost {result['lost']}, moved over 1 {result['moved_over_no_effect']}, over 2 "
          f"{result['moved_over_effect']}, largest move {'-' if largest is None else f'{largest:.2f}'} -> "
          f"{result['outcome']}{' (decides)' if rung == check_rr.DECIDING_RUNG else ''} | without a belt: "
          f"{twelfth['outcome']}")
phone = RUN['iphone']
print(f"\nThe iPhone recordings: {phone['recordings']}, box found {phone['box_found']}, read {phone['read']}, "
      f"rows below the box median {phone['rows']['median']:.0f} ({phone['rows']['min']}-{phone['rows']['max']}); "
      f"resp_ok {phone['resp_ok']}: read {phone['resp_ok_read']}, within 2 of the belt "
      f"{phone['resp_ok_within_2_of_belt']}; within 2 of the stored frames' reading {phone['within_2_of_stored']}")

misses = scored[scored.a_rate.notna() & ((scored.a_rate - scored.belt).abs() > check_rr.WITHIN_BPM)]
low = int((misses.a_share < cuts[0]).sum())
HYPOTHESES = [
    ('H1, A confirms: at least 46 of 56 within 2', a['within_2'] >= 46, f"{a['within_2']} of {a['read']} read"),
    ('H2, A reads all 56', a['read'] == len(scored), f"{a['read']} of {len(scored)}"),
    ('H3, no effect at R2', None if ladder_outcome == 'not assessed' else ladder_outcome == 'no effect', ladder_outcome),
    ("H4, at least half of A's misses below the lowest concentration cut",
     None if misses.empty else 2 * low >= len(misses), f'{low} of {len(misses)} misses'),
    ('H5, the iPhone within 2 of the belt in at least 8 of the 11', phone['resp_ok_within_2_of_belt'] >= 8,
     f"{phone['resp_ok_within_2_of_belt']} of {phone['resp_ok']}"),
]
print('\nHypotheses, on point estimates:')
for name, held, detail in HYPOTHESES:
    verdict = 'untested' if held is None else 'consistent' if held else 'contradicted'
    print(f'  {name}: {verdict} ({detail})')

  A pixel components: read 56 of 56 (100.0%) | within 2 54 (96.4%, 95% 87.9%-99.0%) | within 1 51 (91.1%) | mean abs error 0.53, bias -0.33, limits of agreement -4.98 to +4.32
      McNemar against the baseline: readout only 32, baseline only 0, p = 4.7e-10 -> goes forward
  B vertical shift, described: read 55 of 56 (98.2%) | within 2 54 (98.2%, 95% 90.4%-99.7%) | within 1 53 (96.4%) | mean abs error 0.27, bias -0.09, limits of agreement -1.86 to +1.69
      McNemar against the baseline: readout only 32, baseline only 0, p = 4.7e-10 -> goes forward
A against B, both read in 55: within 2 of each other 53 | both right 53, A only 1, B only 1, neither 0

Within 2 breaths/min among those read (right/read, of n), described
  site group  CH* n 19: A 18/19, B 17/18 | CRC/SCH/BET n 20: A 20/20, B 20/20 | MOL/RING n 17: A 16/17, B 17/17
  Fitzpatrick 1-2 n 16: A 15/16, B 16/16 | 3-4 n 18: A 18/18, B 18/18 | 5-6 n 22: A 21/22, B 20/21
  age         <=40 n 17: A 15/17, B 16/16 | 41-60 n 30: A 30/

## 4. Findings (2026-11-06)
- **Confirmed.** Readout A, run unchanged on VV-60's 56 recordings with a confirmed belt, read all 56 and put 54 within 2 breaths/min of the belt (96.4%, 95% 87.9-99.0%), against 22 for the constant at the training median (39.3%). On the discordant recordings A was right 32 times where the constant was wrong and never the reverse (exact McNemar p = 4.7e-10). The lower bound clears the bar's 70%, coverage is 100%, and the outcome is *confirmed*. Within 1 breath/min: 51 (91.1%); mean absolute error 0.53 breaths/min, bias -0.33, 95% limits of agreement -4.98 to +4.32, their width set by the two misses.
- **As in development.** VV-240 gave 94.7% within 2 breaths/min (95% 90.8-97.0%), and VV-60's 96.4% lies inside that interval. It is about as often as ECG-derived respiration agrees with the same belt on these 56 recordings (94.6%, NB_P3_39 C0).
- **The two misses** sit in A's least concentrated band: below VV-240's lowest quintile cut of concentration (0.561) A is right in 8 of 10, above it in 46 of 46, as H4 expected. Both participants are aged 40 or under; one breathes at 8 breaths/min or fewer, the other at 21-24.
- **A browser-grade encode does not move the reading.** The self-test passed on all 12 raw recordings: their torso regions lie a median 0.26 grey level from the stored frames' (at most 0.32), and R0's readings within 0.01 breaths/min of the stored frames'. At R2 - H.264 at the app's 0.651 bits per pixel per frame - all 11 `resp_ok` recordings kept their reading, within 0.02 breaths/min. No rung moved a reading by more than 0.08 breaths/min, VP8 and a quarter of the rate included, and neither did the twelfth recording at any rung. The ladder's outcome is *no effect*.
- **B, described.** The vertical shift read 55 of 56 and put 54 within 2 breaths/min (within 1: 53; mean absolute error 0.27, 95% limits of agreement -1.86 to +1.69). A and B were right together in 53 of the 55 both read, and each alone in 1. B became a description at the gate, on the development figures, where A led on error and at fast rates; this run does not reopen that choice.
- **Across the strata.** By site group, Fitzpatrick pair and age band the lowest is 15 of 17 (aged 40 or under). By belt rate: 2 of 3 at 8 breaths/min or fewer, 1 of 1 at 9-11, 35 of 35 at 12-20, 12 of 13 at 21-24 and 4 of 4 at 25 or more - too few at either end for a claim. The 2 recordings with a surrogate but no confirmed belt read within 2 breaths/min of it.
- **The iPhone recordings.** The app's face box, padded as the face crop needs it, ends so low in the close-up portrait frame that a median 4.5 of 240 rows remain below it (0-49), and 6 of the 12 keep the 8 rows A needs. All 6 read within 2 breaths/min of the stored frames' reading, and the 5 of them with a confirmed belt within 2 breaths/min of the belt. H5 is contradicted by framing rather than by accuracy: a close-up portrait leaves too little torso below the padded face box.
- **Hypotheses.** H1, at least 46 of 56: consistent (54). H2, all 56 read: consistent. H3, no effect at R2: consistent. H4, the misses below the lowest concentration cut: consistent (2 of 2). H5, the iPhone within 2 breaths/min of the belt in at least 8 of 11: contradicted (5 of 11).

The sentence, as Part 0 fixed it and the run filled it:

> Respiratory rate from the motion of the torso, seated adults at rest. On the 56 of 60 held-out VitalVideos-WW recordings whose waist belt an independent surrogate confirms (lossless lab video with the chest in frame, not a browser capture), readout A reads the breathing rate within 2 breaths/min of the belt often enough to clear a bar fixed in advance: 54 of the 56 recordings it reads (96.4%, 95% interval 87.9-99.0%; within 1 breath/min 91.1%), coverage 100.0%, against 39.3% for a constant at the training median (exact McNemar p = 4.7e-10). A browser-grade encode (H.264 at 0.65 bits per pixel per frame, 11 recordings) did not move the reading. Developed on 208 training recordings (94.7% within 2 breaths/min), where no face-only readout beat the constant. Research use only: not a medical device, and no clinical reading of the rate.

## 5. Conclusion (2026-11-06)
- **What is shown.** On VitalVideos-WW's held-out lab recordings, with the chest in frame and the participants seated and still, the breathing rate read from the motion of the torso below the face lands within 2 breaths/min of a waist belt in 54 of 56 recordings, clearing a bar fixed before any held-out frame was read for breathing; an encode at the app's bit rate leaves the reading where it was.
- **What is not shown.** That the app measures breathing rate: it keeps only the 72x72 face crop, and none of the face's own signals beat a constant (NB_P3_39). Nor that the readout holds for people who move, for framings without the chest, for a browser's own encoder, or at the slow and fast ends of the range: VV-60 holds 3 recordings at 8 breaths/min or fewer and 4 at 25 or more. `resp_ok` leans young (on VV-60 it holds 17 of 18 participants aged 40 or under, 30 of 30 aged 41-60 and 9 of 12 over 60), and site and skin type are confounded in this set. Not a medical device, and no clinical reading of the rate.
- **What follows, as Part 0 fixed it.** With a confirmed verdict and no effect at R2, the capture question - whether the app keeps a whole-frame stream beside the face crop - opens in the app's own work, and an independent webcam-grade test (COHFACE) is considered there. The iPhone recordings mark the framing such a capture needs: the torso below a padded face box, which a close-up portrait leaves out.